# Hyperscanning EEG and BIDS: A Practical Proposal

**Author:** Anne Monnier — Université de Montréal  
**Contribution to:** [BIDS Issue #402](https://github.com/bids-standard/bids-specification/issues/402)

---

## Context

This notebook is intended for anyone doing **multimodal hyperscanning** using:
- A **LabRecorder/LSL** setup that produces XDF files with synchronized EEG streams
- **Behavioral video** recordings (multiple cameras)
- **Subjective ratings** of felt togetherness (e.g. IOS — Inclusion of the Other in the Self scale)

The synthetic dataset used here mirrors the structure of a real clinical research project recording **mother-child dyads** (autistic and non-autistic children) during naturalistic social interaction. The data is synthetic but the architecture is directly inspired by this real-world use case.

---

## The BIDS challenge

| Community proposal | Source | Limitation |
|--------------------|--------|------------|
| `ses-dyadic1` | Issue #402, BIDS FAQ | ❌ Conflicts with longitudinal sessions |
| `acq-dyad001` | Neurostars 2023 (Rémi Gau) | ⚠️ Dyad metadata not in the filename — still requires joining `participants.tsv` for role/group info, making PyBIDS queries no simpler than our proposal |
| **`dyad_id` in `participants.tsv`** | **This proposal** | ✅ Fully standard BIDS — simpler and more expressive |

---

## Experimental protocol (10 tasks)

| Task | Duration |
|------|----------|
| 01 Rest eyes open | 1 min |
| 02 Rest eyes closed | 1 min |
| 03 Spontaneous imitation | 2 min |
| 04 Verbal (day planning) | 2 min |
| 05 Rest eyes open | 1 min |
| 06 Rest eyes closed | 1 min |
| 07 Verbal (day planning) | 2 min |
| 08 Spontaneous imitation | 2 min |
| 09 Rest eyes open | 1 min |
| 10 Rest eyes closed | 1 min |

In [ ]:
import os
from pathlib import Path

# Move to repo root (notebook is in notebooks/)
repo_root = Path().resolve().parent
os.chdir(repo_root)
print(f'Working directory: {repo_root}')

## Part 1 — Source data structure

In [ ]:
from pathlib import Path

def print_tree(path, prefix='', max_depth=3, depth=0):
    if depth > max_depth:
        return
    items = sorted(Path(path).iterdir())
    for i, item in enumerate(items):
        connector = '└── ' if i == len(items)-1 else '├── '
        size = f'  ({item.stat().st_size} B)' if item.is_file() else ''
        print(prefix + connector + item.name + size)
        if item.is_dir():
            ext = '    ' if i == len(items)-1 else '│   '
            print_tree(item, prefix + ext, max_depth, depth+1)

print('sourcedata/')
print_tree('sourcedata')

## Part 2 — The XDF file: source of temporal truth

The XDF file contains all streams in a **shared LSL time reference** — this guarantees synchronization between the two EEG recordings.

**Temporal reference — our position:**  
The synchronization between sub-001 and sub-002 is **implicit** — both share the same `dyad_id` in `participants.tsv`, and the source XDF file in `sourcedata/dyad-001/EEG/` serves as the temporal reference. No additional BIDS field is needed.

In [ ]:
from pathlib import Path

xdf_path = Path('sourcedata/dyad-001/EEG/dyad-001_eeg.xdf')
print(f'XDF file: {xdf_path.name}')
print(f'Size: {xdf_path.stat().st_size} bytes')
print()
print('Expected streams (from LabRecorder/LSL):')
print(f'  {"EGINetAmp_51":25s}  type={"EEG":10s}  channels=128  → mother EEG')
print(f'  {"EGINetAmp_52":25s}  type={"EEG":10s}  channels=128  → child EEG')
print(f'  {"SCAALE_Triggers":25s}  type={"Markers":10s}  channels=1    → task triggers')
print()
print('→ All streams share the same LSL timestamp space.')
print('→ Temporal synchronization between sub-001 and sub-002 is implicit:')
print('  both share dyad_id in participants.tsv + source XDF in sourcedata/.')
print('  No additional BIDS field is needed.')

## Part 3 — IOS ratings

IOS = Inclusion of the Other in the Self (Aron et al. 1992), Likert 1–7, collected **after each task**.  
**Our proposal:** store as a custom column in `events.tsv` — not in `phenotype/` (which is for stable participant traits).

In [ ]:
import pandas as pd

ios_m = pd.read_csv('sourcedata/dyad-001/IOS/ios_sub-001.tsv', sep='\t')
ios_c = pd.read_csv('sourcedata/dyad-001/IOS/ios_sub-002.tsv', sep='\t')

merged = ios_m[['label','value']].rename(columns={'value':'mother_IOS'}).merge(
    ios_c[['label','value']].rename(columns={'value':'child_IOS'}), on='label'
)
print('IOS ratings per task — dyad-001:')
print(merged.to_string(index=False))

## Part 4 — BIDS rawdata structure

In [ ]:
print('rawdata/ (first subject only for clarity)')
print_tree('rawdata/sub-001', max_depth=3)
print('\n... same structure for sub-002 to sub-006')
print()
print('Note: rawdata/ contains no video files.')
print('Videos remain in sourcedata/ — see Part 9 for explanation.')

## Part 5 — Our proposal: `dyad_id` in `participants.tsv`

In [ ]:
participants = pd.read_csv('rawdata/participants.tsv', sep='\t')
print('rawdata/participants.tsv')
print('=' * 60)
print(participants.to_string(index=False))
print()
print('dyad_id is a standard BIDS custom column.')
print('No .bidsignore needed. Validator compliant.')

## Part 6 — Querying dyads

In [ ]:
# Get participants of a dyad
def get_dyad(df, dyad_id):
    dyad = df[df['dyad_id'] == dyad_id]
    return dyad[dyad['role']=='mother'].iloc[0], dyad[dyad['role']=='child'].iloc[0]

mother, child = get_dyad(participants, 'dyad-002')
print(f'dyad-002:')
print(f'  Mother: {mother.participant_id} ({mother.group})')
print(f'  Child:  {child.participant_id} ({child.group})')
print()

# Get all autistic dyads
autistic = participants[participants['group']=='autistic']['dyad_id'].unique()
print(f'Autistic dyads: {list(autistic)}')
print()

# Scalable to any group size
print('All dyads (scalable to triads and larger groups):')
for dyad_id, grp in participants.groupby('dyad_id'):
    members = ' + '.join(f"{r.participant_id} ({r.role})" for _, r in grp.iterrows())
    print(f'  {dyad_id}: {members}')

## Part 7 — Events.tsv with IOS ratings

In [ ]:
# Show all 10 tasks for sub-001
all_events = []
for task_n in range(1, 11):
    task_label = [
        'restEyesOpen','restEyesClosed','imitation','verbal','restEyesOpen',
        'restEyesClosed','verbal','imitation','restEyesOpen','restEyesClosed'
    ][task_n-1]
    fname = f'rawdata/sub-001/ses-01/eeg/sub-001_ses-01_task-{task_n:02d}{task_label}_events.tsv'
    try:
        df = pd.read_csv(fname, sep='\t')
        all_events.append(df)
    except FileNotFoundError:
        pass

all_events_df = pd.concat(all_events, ignore_index=True)
print('events.tsv — sub-001, all 10 tasks:')
print(all_events_df[['onset','duration','trial_type','IOS_rating']].to_string(index=False))
print()
print('IOS_rating is stored directly in events.tsv as a custom column.')
print('This links the subjective rating to its task — our proposed convention.')

## Part 8 — Why not the other proposals?

### `ses-dyadic1` — breaks longitudinal designs
```
# You want 3 timepoints:
sub-001/ses-01/  ← T1
sub-001/ses-02/  ← T2 (6 months)
sub-001/ses-03/  ← T3 (12 months)

# But with ses-dyadic1, ses is already used for the dyad → conflict!
```

### `acq-dyad001` — semantic misuse + no PyBIDS advantage
`acq` is defined for **acquisition parameters** (resolution, sequence type).  
Dyad membership describes *who* was recorded together — not *how*.  
In practice: rare conflict, but semantically incorrect.

More importantly: **it brings no PyBIDS advantage**. Dyad metadata (role, group) is not encoded in the filename — you still need to join `participants.tsv` to get that information. So `acq-dyad001` adds complexity in filenames without simplifying queries:

```python
# With acq-dyad001: you still need participants.tsv for role/group
files = layout.get(acq='dyad001')  # finds files but no metadata
participants = pd.read_csv('participants.tsv', sep='\t')  # still needed!

# With dyad_id in participants.tsv: one step
dyad = participants[participants['dyad_id'] == 'dyad-001']  # role + group + sub IDs
```

### `dyad_id` in `participants.tsv` ✅
Same pattern as `group`, `age`, `sex` — already standard.  
Works longitudinally. Scales to triads. Carries metadata. Simpler PyBIDS queries.

## Part 9 — Open questions for the community

| Question | Our position | Status |
|----------|-------------|--------|
| Post-task ratings (IOS) | Custom column in `events.tsv` — shown in this notebook | ✅ Proposed |
| Shared temporal reference (XDF) | Implicit via `dyad_id` + `sourcedata/` — no new field needed | ✅ Resolved |
| Dyad-level derivatives (PLV) | Local analysis does not require BIDS derivatives. For cross-lab sharing, a future BEP would be valuable. | ⚠️ Future BEP |
| Behavioral video (3 cameras) | `sourcedata/` for raw. Task-segmented videos in `derivatives/video-segmented/dyad-001/` | ⚠️ See below |
| Post-hoc annotations (leader/follower) | `derivatives/video-annotation/dyad-001/` | ⚠️ See below |

### On behavioral video and dyad-level derivatives

Raw videos (3 GoPro cameras per dyad) remain in `sourcedata/` — no BIDS BEP exists for behavioral video.

Task-segmented videos and annotations go into `derivatives/` with a dyad-level directory:
```
derivatives/
  video-segmented/dyad-001/
    dyad-001_task-03imitation_cam-dyad.mp4      ← shared between both participants
    dyad-001_task-03imitation_cam-mother.mp4
    dyad-001_task-03imitation_cam-child.mp4
  openpose/
    sub-001/motion/sub-001_ses-01_task-03imitation_motion.tsv  ← BIDS Motion ✅
    sub-002/motion/sub-002_ses-01_task-03imitation_motion.tsv  ← BIDS Motion ✅
  video-annotation/dyad-001/
    dyad-001_task-03imitation_leader-follower.tsv  ← dyad-level ⚠️
```

**Why `derivatives/` and not `rawdata/`?**  
These data are **shared between both participants** — they cannot belong to `sub-001/` or `sub-002/` alone. Since `rawdata/` has no dyad-level directory in BIDS, `derivatives/` is the only available space. This applies to all shared dyadic outputs: segmented videos, inter-brain connectivity metrics, and leader/follower annotations.

Note that per-subject pose vectors (OpenPose output) go into `sub-XXX/motion/` — fully BIDS Motion compliant ✅. Only the dyad-level outputs fall outside current BIDS conventions.

---

## References

- Poldrack et al. (2024). https://doi.org/10.1162/imag_a_00103  
- Pernet et al. (2019). https://doi.org/10.1038/s41597-019-0104-8  
- Appelhoff et al. (2019). https://doi.org/10.21105/joss.01896  
- Luke et al. (2025). https://www.nature.com/articles/s41597-024-04136-9  
- Monnier et al. (2025). https://doi.org/10.1093/nc/niaf052  
- BIDS Issue #402: https://github.com/bids-standard/bids-specification/issues/402
- Neurostars discussion: https://neurostars.org/t/bids-structure-for-longitudinal-dyadic-data/26173
- NeuroBlueprint Issue #4: https://github.com/neuroinformatics-unit/NeuroBlueprint/issues/4